<a href="https://colab.research.google.com/github/jdmartinev/ST1613-AppliedML-/blob/main/Semana07/Harry_Potter_Expert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain
!pip install -q torch
!pip install -q transformers
!pip install -q sentence-transformers
!pip install -q datasets
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install tqdm
!pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 4.1 MB/s eta 0:00:00


In [ ]:
from langchain.document_loaders import HuggingFaceDatasetLoader #carga de datos de Hugging Face
from langchain.text_splitter import RecursiveCharacterTextSplitter #Division de textos manteniendo contexto
from langchain.embeddings import HuggingFaceEmbeddings #generación de embebings para los textos
from langchain.vectorstores import FAISS #busquedas de similitud
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from transformers import AutoTokenizer, pipeline
from langchain import HuggingFacePipeline
from langchain.chains import RetrievalQA
import torch
from langchain import PromptTemplate, LLMChain

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline,
    AutoModelForCausalLM
    ) #transformers construcción de pipelines personalizados con recuperacion de información


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

def print_lines(text, max_ch = 50):  #division de textos en lineas de 50 caracters max
  current_line = ""                  #Esto solo facilita la lectura
  words = text.split()
  i = 0
  while i < len(words):
    if len(current_line) > 50:
      print(current_line)
      current_line = ""
    else:
      current_line += f"{words[i]} "
      i+=1
  if current_line:
    print(current_line)

cuda


# Datasets

In [ ]:
!wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1YcNlffQl6E09Erst__EZvlC0iJRgy6LF' -O libros.zip
!unzip libros.zip -d ./libros

--2024-10-17 19:46:26--  https://docs.google.com/uc?export=download&id=1YcNlffQl6E09Erst__EZvlC0iJRgy6LF
Resolving docs.google.com (docs.google.com)... 173.194.174.102, 173.194.174.139, 173.194.174.101, ...
Connecting to docs.google.com (docs.google.com)|173.194.174.102|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1YcNlffQl6E09Erst__EZvlC0iJRgy6LF&export=download [following]
--2024-10-17 19:46:26--  https://drive.usercontent.google.com/download?id=1YcNlffQl6E09Erst__EZvlC0iJRgy6LF&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 108.177.125.132, 2404:6800:4008:c01::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|108.177.125.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 20809441 (20M) [application/octet-stream]
Saving to: ‘libros.zip’

libros.zip          100%[===================>]  19.84M  39.2MB/s   

## Leer documentos

In [ ]:
import os
path_docs = "./libros"

os.listdir(path_docs)

['Harry Potter - Book 1 - The Sorcerers Stone.pdf',
 'Harry Potter - Book 6 - The Half-Blood Prince.pdf',
 'Harry Potter - Book 3 - The Prisoner of Azkaban.pdf',
 'Harry Potter - Book 7 - The Deathly Hallows.pdf',
 'Harry Potter - Book 2 - The Chamber of Secrets.pdf',
 'Harry Potter - Book 4 - The Goblet of Fire.pdf',
 'Harry Potter - Book 5 - The Order of the Phoenix.pdf']

In [ ]:
list_pdf = [x for x in os.listdir(path_docs) if x.endswith(".pdf")]

In [ ]:
# Document Transformers
from tqdm.notebook import tqdm
from langchain.document_loaders import PyPDFLoader #cargar y leer archivos PDF.

chunk_size = 1000 # longitud máxima de cada fragmento de texto en caracteres.
chunk_overlap = 200

all_docs = []
text_splitter = RecursiveCharacterTextSplitter(  #divide el texto en fragmentos de tamaño determinado.
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
    )

for name in tqdm(list_pdf):#recorrer todos los archivos pdf
  if name.endswith(".pdf"): #filtro adicicional para tomar solo archivos que finalicen en .pdf
    path_tmp = os.path.join(path_docs, name) #construcción de la ruta completa
    loader = PyPDFLoader(path_tmp) #carga el archivo pdf
    text = loader.load() #extracción del texto del pdf
    chunks_of_text = text_splitter.split_documents(text) # aplicación de división por fragmentos
    all_docs.extend(chunks_of_text) #agrega los fragmentos a la lista


  0%|          | 0/7 [00:00<?, ?it/s]

In [ ]:
from pprint import pprint
chunk = all_docs[100]

print_lines(chunk.page_content)

pprint(chunk.metadata)

lot like yer dad, but yeh’ve got yer mom’s eyes.” Uncle 
Vernon made a funny rasping noise. “I demand that you 
leave at once, sir!” he said. “You are breaking and 
entering!” “Ah, shut up, Dursley, yeh great prune,” 
said the giant; he reached over 
{'page': 36,
 'source': './libros/Harry Potter - Book 1 - The Sorcerers Stone.pdf'}


# Database

## Embeddings

In [ ]:
def get_embeddings_model(model_path=None):
  modelPath = "sentence-transformers/all-MiniLM-l6-v2" #modelo de embeddings
  device = torch.device(
      "cuda" if torch.cuda.is_available() else "cpu"
      )
  #device = "cpu"
  if model_path:
    modelPath = model_path
  model_kwargs = {'device':device} #device sobre el que corre el modelo
  encode_kwargs = {'normalize_embeddings': False} #los espcaios embebidos no se normalizan automaticamente

  embeddings = HuggingFaceEmbeddings(
      model_name=modelPath,     # Provide the pre-trained model's path
      model_kwargs=model_kwargs, # Pass the model configuration options
      encode_kwargs=encode_kwargs # Pass the encoding options
  )
  print(f"device: {device}")
  return embeddings


In [ ]:
embeddings = get_embeddings_model()

device: cuda


In [ ]:
text = "This is a test document."
query_result = embeddings.embed_query(text)
query_result

384

## Vectore Store

In [ ]:
print(len(all_docs))

8927


In [ ]:
db = FAISS.from_documents(all_docs, embeddings) # toma los documentos y sus representaciones en embeddings, y construye un índice que permite realizar búsquedas de manera eficiente.

In [ ]:
question = "Who is Mr. Dursley?"
searchDocs = db.similarity_search(question) #realizar busqueda de similitud por distancia
print_lines(searchDocs[0].page_content) #searchDocs documentos que tienen contenido similar al embedding de la pregunta. Se imprime el contenido del primer documento (el más relevante)

CHAPTER ONE THE BOY WHO LIVED M r. and Mrs. Dursley, 
of number four, Privet Drive, were proud to say that 
they were perfectly normal, thank you very much. They 
were the last people you’d expect to be involved in 
anything strange or mysterious, because they just didn’t 
hold with such nonsense. Mr. Dursley was the director 
of a firm called Grunnings, which made drills. He was 
a big, beefy man with hardly any neck, although he 
did have a very large mustache. Mrs. Dursley was thin 
and blonde and had nearly twice the usual amount of 
neck, which came in very useful as she spent so much 
of her time craning over garden fences, spying on the 
neighbors. The Dursleys had a small son called Dudley 
and in their opinion there was no finer boy anywhere. 
The Dursleys had everything they wanted, but they also 
had a secret, and their greatest fear was that somebody 
would discover it. They didn’t think they could bear 
it if anyone found out about the Potters. Mrs. Potter 
was Mrs. 


In [ ]:
import re
def get_context(db, question, top_k = 2): #devuelve los 2 mejores resultado de la busqueda en la base de datos
  searchDocs = db.similarity_search(question, k= top_k)
  return re.sub(r"\t+", " ", "\n".join([x.page_content for x in searchDocs])) #remplazar tabulación

In [ ]:
get_context(db, question)

'CHAPTER ONE\n \nTHE BOY WHO LIVED\n \n \nM \nr. and Mrs. Dursley, of number four, Privet Drive, were proud to say\nthat they were perfectly normal, thank you very much. They were the last people\nyou’d expect to be involved in anything strange or mysterious, because they just\ndidn’t hold with such nonsense.\n Mr. Dursley was the director of a firm called Grunnings, which made\ndrills. He was a big, beefy man with hardly any neck, although he did have a\nvery large mustache. Mrs. Dursley was thin and blonde and had nearly twice the\nusual amount of neck, which came in very useful as she spent so much of her\ntime craning over garden fences, spying on the neighbors. The Dursleys had a\nsmall son called Dudley and in their opinion there was no finer boy anywhere.\n The Dursleys had everything they wanted, but they also had a secret, and\ntheir greatest fear was that somebody would discover it. They didn’t think they\ncould bear it if anyone found out about the Potters. Mrs. Potter was M

# Large Language Model (LLM)

## Modelo a usar

In [ ]:
import torch
from transformers import AutoModelForQuestionAnswering
from transformers import TFAutoModelForQuestionAnswering
models_hf = ["deepset/minilm-uncased-squad2"] #modelo optimizado de question answering
model_ckpt = models_hf[0] #la posición 0 es la referencia del modelo

In [ ]:
model = AutoModelForQuestionAnswering.from_pretrained(model_ckpt)#carga del modelo pre enttrenado
tokenizer = AutoTokenizer.from_pretrained(model_ckpt) d#a formato adecuado para que el modelo lo interprete


config.json:   0%|          | 0.00/477 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Some weights of the model checkpoint at deepset/minilm-uncased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/107 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
question = "Who is Mr. Dursley?"
context = get_context(db, question)
inputs = tokenizer(question, context, return_tensors="pt")#aplica tokenización
pipe = pipeline("question-answering", model=model, tokenizer=tokenizer)#definición de PipeLine para la tarea de pregunta-respuesta usando el modelo y el tokenizador cargados
pipe(question=question, context=context, topk=3) #aplicación del pipeline para la tarea

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
/usr/local/lib/python3.10/dist-packages/transformers/pipelines/question_answering.py:326: UserWarning: topk parameter is deprecated, use top_k instead
  warnings.warn("topk parameter is deprecated, use top_k instead", UserWarning)


[{'score': 0.3200390636920929,
  'start': 318,
  'end': 357,
  'answer': 'the director of a firm called Grunnings'},
 {'score': 0.13510438799858093,
  'start': 322,
  'end': 357,
  'answer': 'director of a firm called Grunnings'},
 {'score': 0.059819821268320084,
  'start': 318,
  'end': 376,
  'answer': 'the director of a firm called Grunnings, which made\ndrills'}]